# 07 — Merge y concatenación

Combinar DataFrames es una operación fundamental en análisis real: los datos de negocio rara vez viven en un solo archivo.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)


## concat() — apilar DataFrames

`concat` une DataFrames verticalmente (axis=0) o horizontalmente (axis=1). No busca relaciones entre tablas — solo apila.

In [ ]:
# Dividir el dataset y luego reunirlo — simula cargar varios archivos CSV
df_2016 = df[df['Order Date'].str.startswith('2016') if 'Order Date' in df.columns else df.index < 100]

west  = df[df['Region'] == 'West'].copy()
east  = df[df['Region'] == 'East'].copy()
other = df[~df['Region'].isin(['West', 'East'])].copy()

# axis=0 (default) — apila filas
reunido = pd.concat([west, east, other], axis=0, ignore_index=True)
print(f'Original: {len(df)}, Reunido: {len(reunido)}')

# ignore_index=True reindexar desde 0 — evita índices duplicados


In [ ]:
# keys= añade un nivel al índice para saber de qué tabla vino cada fila
combinado = pd.concat(
    [west, east],
    keys=['West', 'East'],
    ignore_index=False
)
print(combinado.index[:5])
print()

# axis=1 — añadir columnas
cols_a = df[['Customer Name', 'Sales']]
cols_b = df[['Region', 'Category']]
lado_a_lado = pd.concat([cols_a, cols_b], axis=1)
print(lado_a_lado.head(3))


## merge() — unir por columna clave

Equivalente al JOIN de SQL. Busca filas que coincidan en una o más columnas.

In [ ]:
# Crear dos DataFrames para el ejemplo
clientes = df[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates('Customer ID')
pedidos  = df[['Order ID', 'Customer ID', 'Sales', 'Category']]

print(f'Clientes: {len(clientes)}, Pedidos: {len(pedidos)}')

# INNER JOIN (default) — solo filas que coinciden en ambas tablas
resultado = pd.merge(pedidos, clientes, on='Customer ID', how='inner')
print(f'Resultado inner: {len(resultado)}')
print(resultado[['Order ID', 'Customer Name', 'Segment', 'Sales']].head())


In [ ]:
# LEFT JOIN — todas las filas de la izquierda, NaN donde no hay match derecho
# RIGHT JOIN — lo contrario
# OUTER JOIN — todas las filas de ambas tablas

# Tabla de segmentos con descuentos (no existe en el dataset original)
descuentos = pd.DataFrame({
    'Segment':    ['Consumer', 'Corporate'],
    'descuento':  [0.05, 0.10]
})

# LEFT JOIN — conserva todos los pedidos aunque el segmento no esté en descuentos
enriquecido = pd.merge(resultado, descuentos, on='Segment', how='left')
print(enriquecido['descuento'].value_counts(dropna=False))
# 'Home Office' quedará con NaN en descuento porque no está en la tabla de descuentos


In [ ]:
# on= cuando la columna tiene el mismo nombre en ambos DataFrames
# left_on / right_on cuando los nombres son distintos

# Ejemplo: la clave se llama 'id_cliente' en una tabla y 'cliente_id' en otra
tabla_a = clientes.rename(columns={'Customer ID': 'id_cliente'})
tabla_b = pedidos.rename(columns={'Customer ID': 'cliente_id'})

resultado2 = pd.merge(
    tabla_b, tabla_a,
    left_on='cliente_id', right_on='id_cliente',
    how='inner'
)
print(resultado2.columns.tolist())


In [ ]:
# suffixes — cuando ambas tablas tienen columnas con el mismo nombre
tabla_ventas_q1 = df[['Customer ID', 'Sales']].rename(columns={'Sales': 'sales_q1'})
tabla_ventas_q2 = df[['Customer ID', 'Sales']].rename(columns={'Sales': 'sales_q2'})

# Si hubiera columnas duplicadas, suffixes las distingue
# pd.merge(a, b, on='key', suffixes=('_izq', '_der'))

# indicator=True añade columna '_merge' que indica de dónde viene cada fila
tabla_x = clientes.head(200)
tabla_y = clientes.tail(200)
con_indicator = pd.merge(tabla_x, tabla_y, on='Customer ID', how='outer', indicator=True)
print(con_indicator['_merge'].value_counts())


---
## Resumen

| Operación | Sintaxis |
|-----------|----------|
| Apilar filas | `pd.concat([df1, df2])` |
| Apilar columnas | `pd.concat([df1, df2], axis=1)` |
| Inner join | `pd.merge(a, b, on='key', how='inner')` |
| Left join | `pd.merge(a, b, on='key', how='left')` |
| Claves distintas | `pd.merge(a, b, left_on='k1', right_on='k2')` |
| Ver origen de cada fila | `pd.merge(..., indicator=True)` |
